In [10]:
!pip install xgboost pandas numpy scikit-learn

In [3]:
import pandas as pd
import xgboost as xgb
import numpy as np
df = pd.read_csv("/Users/mohammadbilal/Documents/Projects/N-Project/output/Dataset.csv")
df = df.drop(columns=["Longitude", "Latitude"], errors="ignore")
print(df.columns)

Index(['Region', 'Target', 'seismic_pga', 'dist_to_water', 'dist_to_fault',
       'population', 'is_protected', 'dist_to_volcano', 'border_distance'],
      dtype='object')


In [13]:
# Dataset is trained on a particular region and tested on a different region, spatial split
test_mask = df["Region"] == "Asia" # Region=Asia is used as test dataset
train_mask = df["Region"] != "Asia"

train_df = df[train_mask]
test_df = df[test_mask]

print(f"Training Dataset size : {len(train_df)} rows")
print(f"Testing Dataset size :     {len(test_df)} rows")


Training Dataset size : 1512 rows
Testing Dataset size :     401 rows


In [14]:
# Removing Geospatial feaatures (lat, lon, region) from the dataset
features = [
    'seismic_pga', 
    'dist_to_water', 
    'dist_to_fault', 
    'population', 
    'is_protected', 
    'dist_to_volcano', 
    'border_distance'
]

X_train = train_df[features]
y_train = train_df['Target']

X_test = test_df[features]
y_test = test_df['Target']

## XGBoost Model

In [ ]:
"""
model = xgb.XGBRegressor(
    objective='reg:squarederror',  # Optimizes for a continuous metric (0.0, 0.5, 1.0)
    n_estimators=150,              # Total number of decision trees
    learning_rate=0.05,            # Controls the step size shrinkage to prevent overfitting
    max_depth=5,                   # Shallow depth keeps decision boundaries smooth
    random_state=42,
    n_jobs=-1                      # Uses all available CPU threads
)
"""

In [ ]:
# Training XGBoost Model 
# Using Grid Search for hyperparameter tuning 
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 150, 200, 300], # Number of Trees
    'learning_rate': [0.005, 0.01, 0.05, 0.1], # Learning Rate
    'max_depth': [3, 5, 7], # Depth of each tree
    'reg_lambda' : [100, 50, 10], # L2 Regularisation 
    'gamma': [0, 2, 5, 10], # Gamma value 
}
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

In [28]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'gamma': 0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'reg_lambda': 50}


In [37]:
model = grid_search.best_estimator_
preds = model.predict(X_test)

In [38]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("-" * 60)
print(f"HOLDOUT TEST REGION: ASIA")
print("-" * 60)
print(f"Mean Absolute Error (MAE):    {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²) Score:         {r2:.4f}")
print("-" * 60)

------------------------------------------------------------
HOLDOUT TEST REGION: ASIA
------------------------------------------------------------
Mean Absolute Error (MAE):    0.1243
Root Mean Squared Error (RMSE): 0.2334
R-squared (R²) Score:         0.7314
------------------------------------------------------------


In [39]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance Score': model.feature_importances_
}).sort_values(by='Importance Score', ascending=False)

print("\nFEATURE IMPORTANCE:")
print(importance.to_string(index=False))


FEATURE IMPORTANCE:
        Feature  Importance Score
     population          0.311819
   is_protected          0.254134
  dist_to_water          0.139663
border_distance          0.134945
  dist_to_fault          0.054315
dist_to_volcano          0.053928
    seismic_pga          0.051195


In [40]:
import joblib
joblib.dump(model, 'xgboost_model.pkl')

['xgboost_model.pkl']

## Light GBM Model

## weigthed Linear Combination

In [ ]:
df = pd.read_csv("/Users/mohammadbilal/Documents/Projects/N-Project/output/Dataset.csv")
df = df.drop(columns=["Longitude", "Latitude"])
df = df.fillna(df.median(numeric_only=True)) #Filling missing values with median of each column

test_mask = df["Region"] == "Asia" 
test_df = df[test_mask]

In [ ]:
# Standardization function to scale features between 0 and 1
# As WLC requires features to be on the same scale

def standardize_min_max(series, invert=False):
    min_val = series.min()
    max_val = series.max()
    
    scaled = (series - min_val) / (max_val - min_val)
    if invert:
        return 1 - scaled
    else:
        return scaled

# Apply standardization based on geographic realities
test_df['is_protected'] = 1 - test_df['is_protected']      
test_df['dist_to_water'] = standardize_min_max(test_df['dist_to_water'], invert=True)  # Close to water is good
test_df['seismic_pga'] = standardize_min_max(test_df['seismic_pga'], invert=True)    # Low PGA is good
test_df['population']   = standardize_min_max(test_df['population'], invert=True)     # Low density near plant is good
test_df['dist_to_fault'] = standardize_min_max(test_df['dist_to_fault'], invert=False) # Far from faults is good
test_df['dist_to_volcano'] = standardize_min_max(test_df['dist_to_volcano'], invert=False) # Far from volcanoes is good
test_df['border_distance'] = standardize_min_max(test_df['border_distance'], invert=False) # Far from borders is good

/var/folders/hj/w1m34kjs5tbgnl01mn92t5pr0000gn/T/ipykernel_25235/2174647753.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['is_protected']    = 1 - test_df['is_protected']
/var/folders/hj/w1m34kjs5tbgnl01mn92t5pr0000gn/T/ipykernel_25235/2174647753.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['dist_to_water']        = standardize_min_max(test_df['dist_to_water'], invert=True)  # Close to water is good
/var/folders/hj/w1m34kjs5tbgnl01mn92t5pr0000gn/T/ipykernel_25235/2174647753.py

In [13]:
# Import weights calculated using AHP method
weights = {
    'is_protected': 0.34,
    'dist_to_water': 0.22,
    'seismic_pga': 0.14,
    'population': 0.14,
    'dist_to_fault': 0.06,
    'dist_to_volcano': 0.06,
    'border_distance': 0.03
}

wlc_predictions = (
    (test_df['is_protected'] * weights['is_protected']) +
    (test_df['dist_to_water'] * weights['dist_to_water']) +
    (test_df['seismic_pga'] * weights['seismic_pga']) +
    (test_df['population'] * weights['population']) +
    (test_df['dist_to_fault'] * weights['dist_to_fault']) +
    (test_df['dist_to_volcano'] * weights['dist_to_volcano']) +
    (test_df['border_distance'] * weights['border_distance'])
)
print("\nWLC PREDICTIONS:")
print(wlc_predictions.head())

y_true = test_df['Target']
print("\nWLC EVALUATION:") 
print(y_true.head())


WLC PREDICTIONS:
8     0.454955
17    0.586982
18    0.586982
19    0.587096
59    0.538110
dtype: float64

WLC EVALUATION:
8     1.0
17    1.0
18    1.0
19    1.0
59    1.0
Name: Target, dtype: float64


In [16]:
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_true, wlc_predictions)
r2 = r2_score(y_true, wlc_predictions)
print("-" * 60)
print("TRADITIONAL WLC PERFORMANCE :")
print("-" * 60)
print(f"Mean Absolute Error (MAE):    {mae:.4f}")
print(f"R-squared (R²) Score:         {r2:.4f}")
print("-" * 60)

------------------------------------------------------------
TRADITIONAL WLC PERFORMANCE :
------------------------------------------------------------
Mean Absolute Error (MAE):    0.4371
R-squared (R²) Score:         -0.0688
------------------------------------------------------------
